# Verification-Centered Software Engineering with Coding Agents

This notebook demonstrates the technical loop taught in the module:

`intent → validated specification → bounded candidate → verifier → evidence/counterexample → revision → scoped claim → engineering approval`

The notebook uses the reproducible prepared Candidate C-017. It does not require an API key or a live coding agent. Its executable scope is candidate verification and revision; architecture, security, policy, and accountable approval remain separate review activities.

Keep four signal classes distinct: **correctness evidence** supports a named property; **revision guidance** suggests what to try next; **model opinion** is a rationale rather than independent evidence; and **approval judgment** decides whether a scoped claim is sufficient for the next lifecycle action.

## 1. Load the candidate with provenance

Loading a candidate is not accepting it. We preserve the source path and candidate ID so another learner can reproduce the evidence.

In [1]:
from pathlib import Path
import importlib.util

cwd = Path.cwd().resolve()
if (cwd / 'exercise_repo').exists():
    materials = cwd
elif (cwd / 'materials' / 'exercise_repo').exists():
    materials = cwd / 'materials'
elif (cwd.parent / 'exercise_repo').exists():
    materials = cwd.parent
else:
    raise RuntimeError('Could not locate the packaged exercise_repo.')

candidate_path = materials / 'exercise_repo' / 'src' / 'discount.py'
spec = importlib.util.spec_from_file_location('candidate_c017', candidate_path)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
candidate = module.promotional_discount
print({'candidate_id': 'C-017', 'source': str(candidate_path.relative_to(materials)), 'state': 'candidate'})

{'candidate_id': 'C-017', 'source': 'exercise_repo/src/discount.py', 'state': 'candidate'}


## 2. Execute the weak verifier

The visible cases check useful examples, but they do not cover every boundary.

In [2]:
visible_cases = [
    ('non-member', 150.0, False, 0.0),
    ('above threshold', 120.0, True, 12.0),
    ('below threshold', 80.0, True, 0.0),
    ('capped', 400.0, True, 25.0),
]

def evaluate_examples(fn, cases):
    rows = []
    for name, total, member, expected in cases:
        actual = fn(total, member)
        rows.append((name, expected, actual, actual == expected))
    return rows

visible_results = evaluate_examples(candidate, visible_cases)
for row in visible_results:
    print(row)
assert all(row[3] for row in visible_results)
print('Weak verifier result: PASS')

('non-member', 0.0, 0.0, True)
('above threshold', 12.0, 12.0, True)
('below threshold', 0.0, 0.0, True)
('capped', 25.0, 25.0, True)
Weak verifier result: PASS


## 3. Ask what would falsify the claim

The validated rule says a member qualifies when the order total is **at least 100.00**. A separately designed boundary case becomes an executable counterexample.

In [3]:
stronger_cases = visible_cases + [
    ('exact threshold', 100.0, True, 10.0),
    ('just below', 99.99, True, 0.0),
]
stronger_results = evaluate_examples(candidate, stronger_cases)
counterexamples = [row for row in stronger_results if not row[3]]
print('Counterexamples:', counterexamples)
assert counterexamples == [('exact threshold', 10.0, 0.0, False)]

Counterexamples: [('exact threshold', 10.0, 0.0, False)]


## 4. Revise and re-run the verifier

For demonstration, the revision is defined locally; the source repository remains unchanged so the lab can be repeated.

In [4]:
def revised_discount(order_total: float, is_member: bool) -> float:
    if order_total < 0:
        raise ValueError('order_total must be non-negative')
    if not is_member or order_total < 100.0:
        return 0.0
    return min(round(order_total * 0.10, 2), 25.0)

revised_results = evaluate_examples(revised_discount, stronger_cases)
assert all(row[3] for row in revised_results)
print('Revised candidate result:', 'PASS', '-', len(revised_results), 'examples')

Revised candidate result: PASS - 6 examples


## 5. A stronger verifier is still not a complete verifier

The business rule rejects negative totals for every customer. A plausible mutation checks membership before validity. It passes the six value examples above but violates the rule for a negative non-member order.

In [5]:
def plausible_mutation(order_total: float, is_member: bool) -> float:
    if not is_member:
        return 0.0
    if order_total < 0:
        raise ValueError('order_total must be non-negative')
    if order_total < 100.0:
        return 0.0
    return min(round(order_total * 0.10, 2), 25.0)

assert all(row[3] for row in evaluate_examples(plausible_mutation, stronger_cases))
try:
    plausible_mutation(-0.01, False)
    mutation_exposed = False
except ValueError:
    mutation_exposed = True
print('Negative non-member case rejects mutation:', mutation_exposed)
assert mutation_exposed is False

Negative non-member case rejects mutation: False


## 6. Write a calibrated claim

A defensible conclusion is scoped. The passing cases below are correctness evidence; the counterexample was revision guidance as well as falsifying evidence; neither is an approval decision:

> Under the recorded Python environment, the revised candidate satisfies the six named value examples. This evidence does not establish all invalid-input, type, floating-point, architecture, security, or production properties. Human approval remains separate.

### Reflection

1. Which part of the first PASS was true, and which conclusion would have been too broad?
2. Who designed each oracle, and was it independent of the candidate?
3. What additional property would you verify before approving production use?